# MD_TP150 — CORRECTED: Filled Data + Actual Expiry

Uses `options_data_filled` (gap-free continuous bars per strike/expiry) with same-strike + same-expiry tracking.

In [ ]:
import duckdb, pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from datetime import timedelta, time
WORK_DIR = Path(r"C:\Users\pc\Downloads\stock hist data\Option Strategy Backtesting")
LOT, DB_PATH = 50, r"C:\Users\pc\Desktop\ucode\project\storage\duckdb\options_history.duckdb"
CUT_TIME = pd.Timestamp("14:15").time()
TABLE = "options_data_filled"  # gap-free continuous data

In [ ]:
# === SPOT ENTRY SIGNALS ===
h1 = pd.read_csv(WORK_DIR / "NIFTY50_ONE_HOUR.csv", parse_dates=["datetime"])
m5 = pd.read_csv(WORK_DIR / "NIFTY50_FIVE_MINUTE.csv", parse_dates=["datetime"])
for d in (h1, m5):
    d["datetime"] = pd.to_datetime(d["datetime"], utc=True).dt.tz_convert("Asia/Kolkata")
    d.sort_values("datetime", inplace=True); d.reset_index(drop=True, inplace=True)
me = m5["datetime"].astype("int64").values
b = (h1["close"]-h1["open"]).abs(); g = h1["close"]>h1["open"]; rr = h1["close"]<h1["open"]
trades = []
for i in range(1, len(h1)):
    if not (rr.iloc[i-1] and g.iloc[i]): continue
    if h1["open"].iloc[i] > h1["close"].iloc[i-1] or h1["close"].iloc[i] < h1["open"].iloc[i-1]: continue
    bval = abs(h1["close"].iloc[i] - h1["open"].iloc[i])
    if bval < abs(h1["close"].iloc[i-1]-h1["open"].iloc[i-1])*0.5 or h1["datetime"].iloc[i].hour == 9: continue
    lv = h1["high"].iloc[i]; ts = h1["datetime"].iloc[i]
    idx = np.searchsorted(me, np.datetime64(ts,"us").astype("int64"), side="right")
    if idx >= len(m5): continue
    bi = idx
    while bi < len(m5) and m5["close"].iloc[bi] <= lv: bi += 1
    if bi >= len(m5)-1: continue
    ri = bi+1
    m5_t = m5["datetime"].dt.time.values
    while ri < len(m5):
        if m5["low"].iloc[ri] < lv and m5["close"].iloc[ri] > lv and m5_t[ri] < CUT_TIME: break
        ri += 1
    if ri >= len(m5): continue
    ep = m5["close"].iloc[ri]
    if ep - m5["low"].iloc[ri] <= 0: continue
    he = ep
    for j in range(ri, len(m5)):
        ca = m5["high"].iloc[j] - m5["low"].iloc[j]
        if m5["high"].iloc[j] > he: he = m5["high"].iloc[j]
        if m5["close"].iloc[j] < he - 55*ca:
            trades.append({"entry_dt": m5["datetime"].iloc[ri], "yr": ts.year, "mo": ts.month})
            break
trades = pd.DataFrame(trades)
trades["ed_naive"] = trades["entry_dt"].dt.tz_localize(None)
trades_pre = trades[trades["ed_naive"] >= pd.Timestamp("2021-06-14")].reset_index(drop=True)
print(f"Spot entries: {len(trades_pre)}")

In [ ]:
# === LOAD ATM DATA (to identify entry strike) ===
con = duckdb.connect(DB_PATH)
df_atm = con.execute(f"""
    SELECT timestamp, close, strike FROM {TABLE}
    WHERE option_type='CALL' AND expiry_code=1 AND expiry_flag='WEEK' AND atm_distance=0
    ORDER BY timestamp
""").fetchdf()
con.close()
df_atm["timestamp"] = pd.to_datetime(df_atm["timestamp"], utc=False)
atm_ts = df_atm["timestamp"].values.astype("datetime64[us]")
atm_cl = df_atm["close"].values.astype(float)
atm_st = df_atm["strike"].values.astype(float)

def lookup_atm(ed):
    ts64 = np.datetime64(ed,"us"); i = np.searchsorted(atm_ts, ts64)
    if i >= len(atm_ts): return len(atm_ts)-1, atm_cl[-1], atm_st[-1]
    if i == 0: return 0, atm_cl[0], atm_st[0]
    return (i, atm_cl[i], atm_st[i]) if atm_ts[i] == ts64 else (i-1, atm_cl[i-1], atm_st[i-1])

In [ ]:
# === LOAD PER-STRIKE CACHE FROM FILLED TABLE (with expiry_date) ===
strike_set = set()
for ed in trades_pre["ed_naive"]: _,_,st = lookup_atm(ed); strike_set.add(int(st))
stk_list = sorted(strike_set)
print(f"Strikes to load: {len(stk_list)}")

con = duckdb.connect(DB_PATH)
df_all = con.execute(f"""
    SELECT timestamp, close, strike, expiry_date
    FROM {TABLE}
    WHERE option_type='CALL' AND expiry_code=1 AND expiry_flag='WEEK'
      AND strike IN ({','.join(map(str,stk_list))})
    ORDER BY strike, expiry_date, timestamp
""").fetchdf()
con.close()
df_all["timestamp"] = pd.to_datetime(df_all["timestamp"], utc=False)
print(f"Loaded {len(df_all)} rows")

# Build cache: strike -> {expiry_dates: [(ts_array, cl_array), ...]}
strike_cache = {}
for stk, grp in df_all.groupby("strike"):
    expiry_map = {}
    for exp_date, exp_grp in grp.groupby("expiry_date", sort=False):
        exp_grp = exp_grp.sort_values("timestamp")
        expiry_map[pd.Timestamp(exp_date).date()] = {
            "ts": exp_grp["timestamp"].values.astype("datetime64[us]"),
            "cl": exp_grp["close"].values.astype(float)
        }
    strike_cache[int(stk)] = expiry_map

print(f"Strikes cached: {len(strike_cache)}")

In [ ]:
# === COMPUTE EXPIRY DATE FOR ENTRY (Thursday rule) ===
def get_weekly_expiry(ts):
    dt = pd.Timestamp(ts)
    days_ahead = (3 - dt.weekday()) % 7
    expiry = dt + timedelta(days=days_ahead)
    if dt.weekday() == 3 and dt.time() >= time(15, 30):
        expiry += timedelta(days=7)
    return expiry.date()

# BUILD TRADE INFOS
trade_infos = []
for idx, row in trades_pre.iterrows():
    ts64 = np.datetime64(row["ed_naive"], "us"); i = np.searchsorted(atm_ts, ts64)
    if i >= len(atm_ts): si = len(atm_ts)-1
    elif i == 0: si = 0
    else: si = i if atm_ts[i] == ts64 else i-1
    st = int(atm_st[si])
    expiry_map = strike_cache.get(st)
    if expiry_map is None: trade_infos.append(None); continue
    
    # Find expiry date matching this entry
    entry_expiry = get_weekly_expiry(row["ed_naive"])
    exp_data = expiry_map.get(entry_expiry)
    if exp_data is None: trade_infos.append(None); continue
    
    s_idx = np.searchsorted(exp_data["ts"], atm_ts[si])
    if s_idx >= len(exp_data["cl"]): trade_infos.append(None); continue
    
    trade_infos.append({"strike": st, "ep": float(exp_data["cl"][s_idx]), "s_idx": int(s_idx),
                        "exp_data": exp_data, "yr": int(row["yr"]), "mo": int(row["mo"]),
                        "entry_ts": exp_data["ts"][s_idx], "expiry": entry_expiry})

matched = sum(1 for t in trade_infos if t is not None)
print(f"Trade infos: {matched}/{len(trade_infos)} matched")

In [ ]:
# === RUN MD_TP150 (filled data, same expiry) ===
TP = 150
records = []
for info in trade_infos:
    if info is None: continue
    ed = info["exp_data"]; s_idx = info["s_idx"]; ep = info["ep"]
    last_idx = len(ed["cl"]) - 1
    if last_idx <= s_idx:
        r = 0; ex_i = s_idx
    else:
        r = None; ex_i = None
        for i in range(s_idx+1, last_idx+1):
            if ed["cl"][i] - ep >= TP:
                r = ed["cl"][i] - ep; ex_i = i; break
        if r is None:
            r = ed["cl"][last_idx] - ep; ex_i = last_idx
    pnl = round(r, 1)
    en = ed["ts"][s_idx]; ex = ed["ts"][ex_i]
    hd = (pd.Timestamp(ex).date() - pd.Timestamp(en).date()).days
    records.append({"entry": en, "exit": ex, "hold_days": hd, "strike": info["strike"],
                    "ep": round(ep,1), "exit_p": round(ed["cl"][ex_i],1),
                    "pnl": pnl, "pnl_rs": int(pnl*LOT), "expiry": str(info["expiry"]),
                    "exit_reason": "TP" if r is not None and ex_i > s_idx and ed["cl"][ex_i]-ep >= TP else "EXP"})

trade_df = pd.DataFrame(records)
trade_df["#"] = range(1, len(trade_df)+1)
print(f"Trades: {len(trade_df)}, max hold: {trade_df['hold_days'].max()} days")

In [ ]:
# === PRINT TRADE BOOK ===
pd.set_option("display.max_rows", 400)
disp = trade_df[["#","entry","exit","hold_days","strike","ep","exit_p","pnl","pnl_rs","expiry","exit_reason"]].copy()
disp["entry"] = disp["entry"].apply(lambda x: pd.Timestamp(x).strftime("%Y-%m-%d"))
disp["exit"] = disp["exit"].apply(lambda x: pd.Timestamp(x).strftime("%Y-%m-%d"))
print(f"TRADE BOOK — MD_TP150 ({len(trade_df)} trades)\n")
display(disp.style.format({"pnl": "{:+.1f}", "pnl_rs": "Rs {:+,.0f}"}))

In [ ]:
# === SUMMARY ===
pnls = trade_df["pnl"].values
n = len(pnls); net = float(pnls.sum())
wr = float((pnls>0).mean()*100); avg = float(pnls.mean())
std = float(pnls.std()) if n>1 else 0
sharpe = float(avg/std*np.sqrt(252)) if std>0 else 0
cum = np.cumsum(pnls); mx = np.maximum.accumulate(cum)
mdd = float((mx-cum).max()) if n>0 else 0
calmar = float(net/mdd) if mdd>0 else 0
pf = float(pnls[pnls>0].sum()/abs(pnls[pnls<0].sum())) if (pnls<0).sum()>0 else 999
tp_cnt = (trade_df["exit_reason"]=="TP").sum()
exp_cnt = n - tp_cnt

print(f"{'='*55}")
print(f"MD_TP150 — FILLED DATA + SAME EXPIRY ✓")
print(f"{'='*55}")
print(f"Trades:          {n}")
print(f"TP hit:          {tp_cnt} ({tp_cnt/n*100:.1f}%)")
print(f"Expiry exit:     {exp_cnt} ({exp_cnt/n*100:.1f}%)")
print(f"Max hold days:   {trade_df['hold_days'].max()}")
print(f"")
print(f"Net PnL:         {net:>+8,.0f} pts  (Rs {net*LOT:>+,.0f})")
print(f"Win Rate:        {wr:.1f}%")
print(f"Avg PnL:         {avg:+.1f} pts")
print(f"Sharpe:          {sharpe:.2f}")
print(f"Calmar:          {calmar:.1f}x")
print(f"Profit Factor:   {pf:.2f}x")
print(f"Max DD:          {mdd:>+8,.0f} pts  (Rs {mdd*LOT:>+,.0f})")

In [ ]:
# === HOLD DAYS + YR/MO BREAKDOWN ===
print("Hold days distribution:")
for d in sorted(trade_df["hold_days"].unique()):
    c = (trade_df["hold_days"]==d).sum()
    print(f"  {d:>2}d: {c} trades")

print(f"\nYEARLY:")
for yr in sorted(trade_df["entry"].apply(lambda x: pd.Timestamp(x).year).unique()):
    p = trade_df[trade_df["entry"].apply(lambda x: pd.Timestamp(x).year)==yr]["pnl"]
    print(f"  {yr}: {len(p)} trades, {p.sum():>+,.0f} pts, {(p>0).mean()*100:.0f}% WR, avg {p.mean():+.1f}")

print(f"\nMONTHLY:")
mo_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
for mo in range(1,13):
    p = trade_df[trade_df["entry"].apply(lambda x: pd.Timestamp(x).month)==mo]["pnl"]
    if len(p): print(f"  {mo_names[mo-1]}: {len(p)} trades, {p.sum():>+,.0f} pts, {(p>0).mean()*100:.0f}% WR")